<a href="https://colab.research.google.com/github/NMinarecioglu/kizilirmak-drought-propagation/blob/main/Trigger_threshold_and_Response_Probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas numpy matplotlib scipy openpyxl seaborn

In [ ]:
"""
Response Probability Contour Plots (Figure 9-10) and
Trigger Threshold Analysis (Figure 13-14)

UPDATED: Date-based event matching (merge_asof) + SPI-RDI primary pair

Requirements:
    pip install pandas numpy matplotlib scipy openpyxl seaborn

Output:
    Response_Probability_v2/ — contour plots
    Trigger_Threshold_v2/    — threshold plots and table
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
EVENTS_FILE = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Run_Theory_NEW/drought_events.csv"
OUTPUT_RESP = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Response_Probability_v2_Yeni_300DPI"
OUTPUT_TRIG = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Trigger_Threshold_v2_Yeni_DPI"

THRESHOLD   = -0.5
STATIONS    = ["Kastamonu", "Sivas", "Kayseri", "Yozgat", "Kirikkale"]
MIN_EVENTS  = 8
MATCH_TOL   = pd.Timedelta("90D")   # matching tolerance

# Analysis pairs — primary: SPI-RDI, secondary: SPI-SDI where data permits
PAIRS = [
    ("SPI", "RDI", [3, 6]),    # strong τ all stations
    ("SPI", "SDI", [3, 6]),    # only Kastamonu + Sivas reliable
]
# ============================================================

os.makedirs(OUTPUT_RESP, exist_ok=True)
os.makedirs(OUTPUT_TRIG, exist_ok=True)

df_all = pd.read_csv(EVENTS_FILE, parse_dates=["Start", "End"])
df_all["Peak"] = df_all["Peak"].abs()
df_ev = df_all[df_all["Threshold"] == THRESHOLD].copy()
print(f"Events loaded: {len(df_ev)}\n")


# ===========================================================
# DATE-BASED EVENT MATCHING
# ===========================================================
def match_events(station, idx_x, idx_y, scale):
    """Match SPI events with RDI/SDI events by nearest Start date."""
    sx = df_ev[(df_ev["Station"]==station) & (df_ev["Index"]==idx_x) &
               (df_ev["Scale"]==scale)][["Start","Duration","Severity","Peak","Intensity"]].copy()
    sy = df_ev[(df_ev["Station"]==station) & (df_ev["Index"]==idx_y) &
               (df_ev["Scale"]==scale)][["Start","Duration","Severity","Peak","Intensity"]].copy()

    sx = sx.rename(columns={"Duration":"D_X","Severity":"S_X","Peak":"P_X","Intensity":"I_X"})
    sy = sy.rename(columns={"Duration":"D_Y","Severity":"S_Y","Peak":"P_Y","Intensity":"I_Y"})

    merged = pd.merge_asof(
        sx.sort_values("Start"),
        sy.sort_values("Start"),
        on="Start", tolerance=MATCH_TOL, direction="nearest"
    ).dropna()

    return merged


# ===========================================================
# COPULA HELPERS
# ===========================================================
def empirical_cdf(x):
    rank = stats.rankdata(x, method="average")
    return rank / (len(x) + 1)

def fit_best_marginal(x):
    x = np.array(x); x = x[x > 0]
    best_ks, best_dist, best_params = np.inf, None, None
    for dist in [stats.gamma, stats.lognorm, stats.genextreme,
                 stats.weibull_min, stats.expon]:
        try:
            params = dist.fit(x, floc=0)
            ks, _  = stats.kstest(x, dist.cdf, args=params)
            if ks < best_ks:
                best_ks, best_dist, best_params = ks, dist, params
        except Exception:
            pass
    return best_dist, best_params

def marginal_cdf(x_val, dist, params):
    return np.clip(dist.cdf(x_val, *params), 1e-6, 1-1e-6)

# Copula CDFs
def frank_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    return -1/t*np.log(1+(np.exp(-t*u)-1)*(np.exp(-t*v)-1)/np.maximum(np.exp(-t)-1,eps))

def frank_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    et=np.exp(-t); eu=np.exp(-t*u); ev=np.exp(-t*v)
    return np.maximum(np.abs(-t*(et-1)*eu*ev/((et-1+(eu-1)*(ev-1))**2+eps)),eps)

def gumbel_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    A=(-np.log(u))**t+(-np.log(v))**t
    return np.exp(-A**(1/t))

def gumbel_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    lu=-np.log(u); lv=-np.log(v)
    A=lu**t+lv**t; C=np.exp(-A**(1/t))
    return np.maximum(C*(u*v)**(-1)*A**(-2+2/t)*(lu*lv)**(t-1)*(1+(t-1)*A**(-1/t)),eps)

def clayton_cdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    return np.maximum(u**(-t)+v**(-t)-1,eps)**(-1/t)

def clayton_pdf(u, v, t):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    A=np.maximum(u**(-t)+v**(-t)-1,eps)
    return np.maximum((1+t)*(u*v)**(-(1+t))*A**(-(2+1/t)),eps)

def gaussian_ll(u, v, rho):
    eps=1e-10; u=np.clip(u,eps,1-eps); v=np.clip(v,eps,1-eps)
    x=stats.norm.ppf(u); y=stats.norm.ppf(v); det=1-rho**2
    return np.sum(-0.5*np.log(det)-(rho**2*(x**2+y**2)-2*rho*x*y)/(2*det))

def gaussian_cdf_scalar(u_s, v_s, rho):
    from scipy.stats import multivariate_normal
    u_s=np.clip(u_s,1e-6,1-1e-6); v_s=np.clip(v_s,1e-6,1-1e-6)
    x=stats.norm.ppf(u_s); y=stats.norm.ppf(v_s)
    return multivariate_normal.cdf([x,y],mean=[0,0],cov=[[1,rho],[rho,1]])

def fit_copula(u, v):
    from scipy.stats import kendalltau
    tau,_ = kendalltau(u,v)
    best  = None

    for name, pdf_fn, bounds, theta0 in [
        ("Frank",   frank_pdf,   (-50,50),    4*tau),
        ("Gumbel",  gumbel_pdf,  (1.001,50),  max(1/(1-tau+1e-5),1.01)),
        ("Clayton", clayton_pdf, (0.01,50),   max(2*tau/(1-tau+1e-5),0.01)),
    ]:
        try:
            res = minimize_scalar(
                lambda t: -np.sum(np.log(np.maximum(pdf_fn(u,v,t),1e-300))),
                bounds=bounds, method="bounded"
            )
            ll  = -res.fun
            aic = 2 - 2*ll
            if best is None or aic < best[3]:
                best = (name, res.x, ll, aic)
        except Exception:
            pass

    # Gaussian
    try:
        res  = minimize_scalar(lambda r: -gaussian_ll(u,v,r),
                               bounds=(-0.999,0.999), method="bounded")
        ll   = -res.fun
        aic  = 2 - 2*ll
        if best is None or aic < best[3]:
            best = ("Gaussian", res.x, ll, aic)
    except Exception:
        pass

    return best   # (family, param, ll, aic)

def copula_cdf(u_val, v_val, family, param):
    u_val=np.clip(u_val,1e-6,1-1e-6); v_val=np.clip(v_val,1e-6,1-1e-6)
    if family=="Frank":   return frank_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Gumbel":  return gumbel_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Clayton": return clayton_cdf(np.array([u_val]),np.array([v_val]),param)[0]
    if family=="Gaussian":return gaussian_cdf_scalar(u_val, v_val, param)
    return u_val * v_val

def conditional_prob_grid(u_grid, v_grid, family, param):
    """P(V<=v | U=u) via numerical derivative dC/du."""
    h  = 1e-4
    prob = np.zeros((len(v_grid), len(u_grid)))
    for j, uv in enumerate(u_grid):
        u1 = np.clip(uv+h, 1e-6, 1-1e-6)
        u2 = np.clip(uv-h, 1e-6, 1-1e-6)
        for i, vv in enumerate(v_grid):
            c1 = copula_cdf(u1, vv, family, param)
            c2 = copula_cdf(u2, vv, family, param)
            prob[i,j] = np.clip((c1-c2)/(u1-u2), 0, 1)
    return prob


# ===========================================================
# FIGURE 9-10: Response Probability Contour
# ===========================================================
def plot_response_probability(idx_x, idx_y, variable, scale):
    matched_data = {}
    for station in STATIONS:
        m = match_events(station, idx_x, idx_y, scale)
        if len(m) >= MIN_EVENTS:
            matched_data[station] = m

    if not matched_data:
        print(f"  No sufficient data for {idx_x}-{idx_y} {variable} Scale-{scale}")
        return

    n_st  = len(matched_data)
    fig, axes = plt.subplots(1, n_st, figsize=(4*n_st, 5), sharey=True)
    if n_st == 1: axes = [axes]
    fig.patch.set_facecolor("white")

    col_x = "D_X" if variable=="Duration" else "S_X"
    col_y = "D_Y" if variable=="Duration" else "S_Y"
    levels = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
    colors_l = ["#4575b4","#74add1","#abd9e9","#e0f3f8",
                "#fee090","#fdae61","#f46d43","#d73027","#a50026"]

    for ax, (station, m) in zip(axes, matched_data.items()):
        x_raw = m[col_x].values.astype(float)
        y_raw = m[col_y].values.astype(float)

        dist_x, par_x = fit_best_marginal(x_raw)
        dist_y, par_y = fit_best_marginal(y_raw)
        if dist_x is None or dist_y is None:
            ax.set_title(f"{station}\nFit failed"); continue

        u = empirical_cdf(x_raw)
        v = empirical_cdf(y_raw)
        cop = fit_copula(u, v)
        if cop is None:
            ax.set_title(f"{station}\nCopula failed"); continue

        family, param, _, aic = cop

        # Frank high-theta numerical fix — replace with next best copula
        if family == "Frank" and abs(param) > 8:
            candidates = []
            for name, pdf_fn, bounds in [
                ("Gumbel",  gumbel_pdf,  (1.001, 50)),
                ("Clayton", clayton_pdf, (0.01,  50)),
            ]:
                try:
                    res = minimize_scalar(
                        lambda t, fn=pdf_fn: -np.sum(np.log(np.maximum(fn(u,v,t),1e-300))),
                        bounds=bounds, method="bounded"
                    )
                    candidates.append((name, res.x, -res.fun, 2+2*res.fun))
                except Exception:
                    pass
            try:
                res2 = minimize_scalar(lambda r: -gaussian_ll(u,v,r),
                                       bounds=(-0.999,0.999), method="bounded")
                candidates.append(("Gaussian", res2.x, -res2.fun, 2+2*res2.fun))
            except Exception:
                pass
            if candidates:
                candidates.sort(key=lambda x: x[3])
                family, param = candidates[0][0], candidates[0][1]
                print(f"    [FIX] {station}: Frank replaced by {family} (theta={param:.3f})")

        tau,_ = stats.kendalltau(x_raw, y_raw)

        x_max = max(x_raw.max()*1.1, 5)
        y_max = max(y_raw.max()*1.1, 5)
        x_grid = np.linspace(0.1, x_max, 25)
        y_grid = np.linspace(0.1, y_max, 25)

        u_grid = np.array([marginal_cdf(xv, dist_x, par_x) for xv in x_grid])
        v_grid = np.array([marginal_cdf(yv, dist_y, par_y) for yv in y_grid])

        prob = conditional_prob_grid(u_grid, v_grid, family, param)

        cs = ax.contour(x_grid, y_grid, prob, levels=levels,
                        colors=colors_l, linewidths=1.5)
        ax.clabel(cs, fmt="%.1f", fontsize=7, inline=True)
        ax.scatter(x_raw, y_raw, s=12, color="black", alpha=0.4, zorder=5)
        ax.set_xlim(0, x_max); ax.set_ylim(0, y_max)
        ax.set_xlabel(f"{idx_x}-{scale} {variable}", fontsize=9)
        ax.set_title(f"{station}\n{family} (τ={tau:.2f}, n={len(m)})",
                     fontsize=9, fontweight="bold")
        ax.grid(linestyle="--", alpha=0.3)

    axes[0].set_ylabel(f"{idx_y}-{scale} {variable}", fontsize=9)
    fig.suptitle(
        f"P({idx_y}-{scale} {variable} ≤ y  |  {idx_x}-{scale} {variable} = x)\n"
        f"Date-matched events  |  Threshold = {THRESHOLD}",
        fontsize=11, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_RESP,
          f"ResponseProb_{idx_x}-{idx_y}_{variable}_Scale{scale}.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")


# ===========================================================
# FIGURE 13-14: Trigger Threshold
# P(Y<=y_thr | x1<X<=x2) iterating over SPI severity
# ===========================================================
def compute_trigger(station, idx_x, idx_y, scale, y_threshold_pct=50):
    m = match_events(station, idx_x, idx_y, scale)
    if len(m) < MIN_EVENTS:
        return None, None

    # Use Intensity (Severity/Duration) — keeps values in SPI-comparable range
    x_raw = m["I_X"].values.astype(float)
    y_raw = m["I_Y"].values.astype(float)

    # Remove invalid intensities
    valid = (x_raw > 0) & (y_raw > 0) & np.isfinite(x_raw) & np.isfinite(y_raw)
    x_raw = x_raw[valid]
    y_raw = y_raw[valid]

    if len(x_raw) < MIN_EVENTS:
        return None, None

    dist_x, par_x = fit_best_marginal(x_raw)
    dist_y, par_y = fit_best_marginal(y_raw)
    if dist_x is None or dist_y is None:
        return None, None

    u = empirical_cdf(x_raw)
    v = empirical_cdf(y_raw)
    cop = fit_copula(u, v)
    if cop is None:
        return None, None

    family, param = cop[0], cop[1]

    # Y threshold: median intensity of Y series
    y_thr = np.percentile(y_raw, y_threshold_pct)
    v_thr = marginal_cdf(y_thr, dist_y, par_y)

    # Iterate X (SPI intensity) from min to max in steps of 0.1
    x_vals = np.arange(0.1, x_raw.max()*1.05, 0.1)
    probs  = []
    x_out  = []

    for xi in x_vals:
        xi0 = max(xi - 0.5, 0.01)
        u2  = marginal_cdf(xi,  dist_x, par_x)
        u1  = marginal_cdf(xi0, dist_x, par_x)
        denom = u2 - u1
        if denom < 1e-8: continue

        c2 = copula_cdf(u2, v_thr, family, param)
        c1 = copula_cdf(u1, v_thr, family, param)
        p  = np.clip((c2-c1)/denom, 0, 1)
        probs.append(p)
        x_out.append(xi)

    return np.array(x_out), np.array(probs)


def plot_trigger_threshold(idx_x, idx_y, scale):
    colors = {"Kastamonu":"#1f77b4","Sivas":"#ff7f0e",
              "Kayseri":"#2ca02c","Yozgat":"#d62728","Kirikkale":"#9467bd"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("white")

    records = []

    # Panel 1: Trigger curves
    ax = axes[0]
    for station in STATIONS:
        xv, pv = compute_trigger(station, idx_x, idx_y, scale)
        if xv is None: continue
        ax.plot(xv, pv, "-o", markersize=4, linewidth=1.8,
                color=colors[station], label=station)
        # Find 50% threshold
        if pv.max() >= 0.5:
            idx50 = np.argmin(np.abs(pv - 0.5))
            ax.axvline(xv[idx50], color=colors[station],
                       linestyle=":", linewidth=0.8, alpha=0.6)
            records.append({"Station":station,"Pair":f"{idx_x}-{idx_y}",
                            "Scale":scale,"Thr_50pct":round(xv[idx50],2),
                            "Max_prob":round(pv.max(),3)})

    ax.axhline(0.5, color="black", linewidth=1.5, linestyle="--",
               label="50% level")
    ax.set_xlabel(f"{idx_x}-{scale} Mean Intensity (Severity/Duration)", fontsize=10)
    ax.set_ylabel(f"P({idx_y} Intensity ≥ median | {idx_x} Intensity in interval)",
                  fontsize=9)
    ax.set_ylim(0, 1); ax.set_xlim(left=0)
    ax.set_title(f"Trigger Probability: {idx_x}-{scale} → {idx_y}-{scale}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(linestyle="--", alpha=0.4)

    # Panel 2: Grade propagation — 4 SPI grades x 4 SDI levels
    ax2 = axes[1]
    grade_bounds = {"Mild":(0.3,0.7),"Moderate":(0.7,1.2),"Severe":(1.2,2.0),"Extreme":(2.0,5.0)}
    grade_levels = {"Mild":0.5,"Moderate":1.0,"Severe":1.5,"Extreme":2.5}
    colors_g = {"Mild":"#74add1","Moderate":"#fdae61",
                "Severe":"#f46d43","Extreme":"#a50026"}
    markers  = {"Mild":"o","Moderate":"s","Severe":"^","Extreme":"D"}

    plotted = False
    for station in STATIONS:
        m = match_events(station, idx_x, idx_y, scale)
        if len(m) < MIN_EVENTS: continue

        x_raw = m["I_X"].values.astype(float)
        y_raw = m["I_Y"].values.astype(float)
        valid = (x_raw>0)&(y_raw>0)&np.isfinite(x_raw)&np.isfinite(y_raw)
        x_raw = x_raw[valid]; y_raw = y_raw[valid]
        if len(x_raw) < MIN_EVENTS: continue
        dist_x, par_x = fit_best_marginal(x_raw)
        dist_y, par_y = fit_best_marginal(y_raw)
        if dist_x is None: continue
        u = empirical_cdf(x_raw); v = empirical_cdf(y_raw)
        cop = fit_copula(u, v)
        if cop is None: continue
        family, param = cop[0], cop[1]

        # Use first valid station for grade plot
        for spi_grade, (lo, hi) in grade_bounds.items():
            u2 = marginal_cdf(hi, dist_x, par_x)
            u1 = marginal_cdf(lo, dist_x, par_x)
            denom = u2 - u1
            if denom < 1e-8: continue
            probs_g = []
            for sdi_grade, sdi_thr in grade_levels.items():
                v_t = marginal_cdf(sdi_thr, dist_y, par_y)
                c2  = copula_cdf(u2, v_t, family, param)
                c1  = copula_cdf(u1, v_t, family, param)
                probs_g.append(np.clip((c2-c1)/denom, 0, 1))
            ax2.plot(list(grade_levels.keys()), probs_g,
                     "-"+markers[spi_grade], color=colors_g[spi_grade],
                     label=f"SPI {spi_grade}", linewidth=1.8, markersize=7)
        ax2.set_title(f"Grade Propagation — {station}\n"
                      f"{idx_x}-{scale} → {idx_y}-{scale}",
                      fontsize=10, fontweight="bold")
        plotted = True
        break   # show one representative station

    if not plotted:
        ax2.text(0.5,0.5,"Insufficient data",ha="center",va="center",
                 transform=ax2.transAxes)

    ax2.set_xlabel(f"{idx_y} Drought Grade", fontsize=10)
    ax2.set_ylabel("Conditional Probability", fontsize=10)
    ax2.set_ylim(0, 1)
    ax2.legend(fontsize=9, title="SPI Drought Grade")
    ax2.grid(linestyle="--", alpha=0.4)

    fig.suptitle(
        f"Trigger Thresholds & Grade Propagation — {idx_x}-{scale} → {idx_y}-{scale}\n"
        f"X-axis: Mean Intensity (Severity/Duration)  |  Threshold = {THRESHOLD}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    out = os.path.join(OUTPUT_TRIG,
          f"TriggerThreshold_{idx_x}-{idx_y}_Scale{scale}.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved -> {out}")
    return records


# ===========================================================
# MAIN
# ===========================================================
print("="*60)
print("  Response Probability & Trigger Threshold  (v2)")
print("="*60+"\n")

all_records = []

for idx_x, idx_y, scales in PAIRS:
    for scale in scales:
        print(f"\n--- {idx_x} → {idx_y}  Scale-{scale} ---")
        for var in ["Duration", "Severity"]:
            print(f"  [{var}] Response probability...")
            plot_response_probability(idx_x, idx_y, var, scale)
        print(f"  [Trigger] Threshold analysis...")
        recs = plot_trigger_threshold(idx_x, idx_y, scale)
        if recs: all_records.extend(recs)

# Save trigger threshold table
if all_records:
    df_trig = pd.DataFrame(all_records)
    df_trig.to_excel(os.path.join(OUTPUT_TRIG,"trigger_threshold_table_v2.xlsx"), index=False)
    df_trig.to_csv( os.path.join(OUTPUT_TRIG,"trigger_threshold_table_v2.csv"),  index=False)
    print("\nTrigger threshold table:")
    print(df_trig.to_string(index=False))

print(f"\nAll done!")
print(f"  Response Prob  -> {OUTPUT_RESP}")
print(f"  Trigger Thresh -> {OUTPUT_TRIG}")